# Step 02 of 04 - Extract date-region crops

## Run step 01 model over the receipt dataset, extract each detected date field by warping them into an upright crop to make step 03 easier to execute.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

## Configuration

Paths and detection parameters. `ML_DIR` resolves to `backend/ml`.

- **`MODEL_PATH`** — Stage-1 `best.pt` weights (update the placeholder path to your run).
- **`STAGE1_DATA`** — source receipt dataset with `images/{train,val,test}`.
- **`OUTPUT_ROOT`** — where extracted date crops are written.
- **`MIN_OUTPUT_HEIGHT`** — crops shorter than this are upscaled.
- **`CONF_THRESHOLD`** / **`PADDING`** — detection confidence cutoff and how much
  the detected quad is expanded before warping.

In [ ]:
ML_DIR = Path.cwd().parent

# === Config ===
# Update model path to weights/best.pt from step 1
MODEL_PATH = ML_DIR / "models/stage_01/path/to/weights/best.pt"
STAGE1_DATA = ML_DIR / "data/stage_01/receipt_dataset"
OUTPUT_ROOT = ML_DIR / "data/stage_02/date_crops"
MIN_OUTPUT_HEIGHT = 64   # if native height is below this, upscale
DATE_CLASS_ID = 0
CONF_THRESHOLD = 0.25 
PADDING = 0.10

## Geometry helpers

Utilities to turn a detected oriented box into a clean crop:

- **`order_corners`** - sorts the 4 detected points into TL, TR, BR, BL order.
- **`expand_quad`** - pads the quad outward by a ratio so the warp keeps context.
- **`warp_crop_native`** - perspective-warps the quad to an upright rectangle at
  its native resolution (no down/upscaling unless the crop is below the minimum
  height).

In [ ]:
def order_corners(pts):
    """Order 4 points as TL, TR, BR, BL based on geometry."""
    pts = np.array(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = pts[:, 1] - pts[:, 0]
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def expand_quad(corners, padding_ratio):
    """Expand a quadrilateral outward by a percentage of its size."""
    center = corners.mean(axis=0)
    expanded = corners + (corners - center) * padding_ratio
    return expanded


def warp_crop_native(img, corners):
    """Perspective-warp to upright rectangle at the source quad's native resolution.
    
    Computes output dimensions from the actual side lengths of the rotated
    quadrilateral, so no upscaling or downscaling occurs — pixels stay at
    their original sampling density.
    """
    src = order_corners(corners)
    
    w_top = np.linalg.norm(src[1] - src[0])
    w_bot = np.linalg.norm(src[2] - src[3])
    h_left = np.linalg.norm(src[3] - src[0])
    h_right = np.linalg.norm(src[2] - src[1])
    
    out_w = int(max(w_top, w_bot))
    out_h = int(max(h_left, h_right))
    
    # If the native resolution is too small for digit detection,
    # upscales uniformly to preserve aspect ratio
    if out_h < MIN_OUTPUT_HEIGHT:
        scale = MIN_OUTPUT_HEIGHT / out_h
        out_h = MIN_OUTPUT_HEIGHT
        out_w = int(out_w * scale)
    
    out_w = max(out_w, 50)
    out_h = max(out_h, 20)
    
    dst = np.array([
        [0, 0],
        [out_w - 1, 0],
        [out_w - 1, out_h - 1],
        [0, out_h - 1]
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    # INTER_CUBIC for smoother upscale than INTER_LINEAR
    return cv2.warpPerspective(img, M, (out_w, out_h), flags=cv2.INTER_CUBIC)

## Process a split

`process_split` runs the detector over every image in one split, keeps the
highest-confidence date detection per image, warps it to a crop, and saves it as
PNG. It also collects crop-size statistics and counts images where no date was
found.

In [ ]:
def process_split(split_name, model):
    images_dir = STAGE1_DATA / 'images' / split_name
    output_dir = OUTPUT_ROOT / split_name
    output_dir.mkdir(parents=True, exist_ok=True)
    
    images = sorted(images_dir.iterdir())
    count = 0
    no_date = 0
    
    sizes = []
    
    for img_path in images:
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        results = model.predict(
            source=str(img_path),
            imgsz=1024,
            conf=CONF_THRESHOLD,
            verbose=False,
        )
        
        result = results[0]
        if result.obb is None or len(result.obb) == 0:
            no_date += 1
            continue
        
        classes = result.obb.cls.cpu().numpy().astype(int)
        confs = result.obb.conf.cpu().numpy()
        corners_all = result.obb.xyxyxyxy.cpu().numpy()
        
        date_indices = np.where(classes == DATE_CLASS_ID)[0]
        if len(date_indices) == 0:
            no_date += 1
            continue
        
        best_idx = date_indices[np.argmax(confs[date_indices])]
        corners = corners_all[best_idx]
        corners_padded = expand_quad(corners, PADDING)
        
        # Native-resolution warp NO upscaling
        crop = warp_crop_native(img, corners_padded)
        sizes.append(crop.shape[:2])
        
        out_path = output_dir / f"{img_path.stem}.png"
        cv2.imwrite(str(out_path), crop)
        count += 1
    
    if sizes:
        heights = [s[0] for s in sizes]
        widths = [s[1] for s in sizes]
        print(f"{split_name}: {count} crops extracted, {no_date} no date")
        print(f"Width:  min={min(widths)}, max={max(widths)}, mean={sum(widths)//len(widths)}")
        print(f"Height: min={min(heights)}, max={max(heights)}, mean={sum(heights)//len(heights)}")
    else:
        print(f"{split_name}: 0 crops extracted")

## Main routine

Loads the model and runs `process_split` over the `train`, `val`, and `test`
splits.

In [ ]:
def main():
    model = YOLO(MODEL_PATH)
    print("Model loaded. Class names:", model.names)
    
    for split in ['train', 'val', 'test']:
        process_split(split, model)
    
    print(f"\nDone. Crops saved to {OUTPUT_ROOT}")

## Run

Execute the full extraction pipeline.

In [ ]:
main()